In [ ]:
from glob import glob
import os
import pandas as pd
import json
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import warnings ; warnings.filterwarnings('ignore')
import numpy as np

import torch
from transformers import AutoModel, AutoTokenizer
from src.utils.preprocessing import Preprocessor

from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib import font_manager as fm

from umap import UMAP
from scipy.stats import f
from tqdm import tqdm

def check_label(df, s):
    df = df.to_frame()
    df['label'] = s
    return df


font_fname = os.environ.get('PROJECT_FONT_PATH', '')  # point at any local CJK/serif .ttf
font_family = fm.FontProperties(fname=font_fname).get_name() #폰트 설정
plt.rcParams["font.family"] = font_family  #폰트 적용



negative_patterns = ['without','not? (sugges|compat)', 'unlike', 'neither', 'no', 'negative']
uncertain_patterns = ['rule out', 'Rule Out', "Rule out"]


target_class = ['negative','uncertain','positive']
cancer_merge_keys = ['Neurologic', 'Hematologic', 'Head & Neck', 'Endocrine']
# merged_cancer_label = 'Neurologic & Hematologic & HeadNeck & Endocrine'
merged_cancer_label = 'Merged'

figdir = os.path.join('.','figure')
os.makedirs(figdir, exist_ok = True)

In [ ]:

rev_dir_path = os.path.join('data','reviewer')
metas_df, recur_df = [pd.read_excel(i, sheet_name=None) for i in sorted(glob(os.path.join(rev_dir_path,'*.xlsx')))]


In [ ]:
agg_df_save_dir = os.path.join('data','reviewed_labels')
load_all = pd.read_excel([i for i in glob(os.path.join(agg_df_save_dir, '*')) if '.xlsx' in i][0])
# cancer_type_df = pd.read_excel(os.path.join('data','cancer_type_map.xlsx'))


In [ ]:
grouped = load_all.groupby(['병원등록번호', '수술일자'])
meta_unique_counts = grouped['Metastasis_class'].nunique().tolist()
recur_unique_counts = grouped['Recurrence_class'].nunique().tolist()
print("Metastasis class 고윳값 갯수 리스트:", meta_unique_counts)
print("Recurrence class 고윳값 갯수 리스트:", recur_unique_counts)

In [ ]:

cancer_dict_path = os.path.join('.', 'cancer_dict.json')
with open(cancer_dict_path, 'r', encoding='utf-8') as file:
    cancer_dict = json.load(file)
cancer_en_dict = {int(k) : v['name_en'] for k, v in cancer_dict.items()}

cancer_merge_keys = ['Neurologic', 'Hematologic', 'Head & Neck', 'Endocrine']
merged_cancer_label = 'Merged'
cnacer_merged_en_dict = {k:v if v not in cancer_merge_keys else merged_cancer_label for k, v in cancer_en_dict.items() }

In [ ]:
cancer_order = list(cancer_en_dict.values()) + ['Missing']

In [ ]:

# cancer_type_df.iloc[:,2].fillna(' ', inplace = True)
# cancer_type_df.iloc[:,3].fillna(missing_numb, inplace = True)
# # cancer_type_df.loc[:,'암번호'] = cancer_type_df.loc[:,'암번호'].astype(int)
# cancer_type_df['암번호'] = cancer_type_df.암번호.astype(int)

def age_group(age):
    for n, (sty, edy) in enumerate(zip(age_bins[:-1], age_bins[1:])):
        if sty <= age < edy:
            return n
    else:
        return n + 1

age_bins = [0] + list(range(40, 90, 10))
print(age_bins)

age_dict = {}
for n, (st, ed) in enumerate(zip(age_bins, age_bins[1:])):
    strs = r'~'
    if n > 0:
        strs = f'{st}{strs}'
    strs = f'{strs}{ed}'
    age_dict[n] = strs
else:
    age_dict[n+1] = f'{ed}~'

age_order = list(age_dict.values())
print(age_dict, age_order)

In [ ]:
merged_all = load_all.copy()
merged_all['index'] = merged_all.index
merged_all['나이구간'] = merged_all['검사나이'].apply(age_group)

In [ ]:
metas_labeled_df = pd.concat([metas_df[c] for c in target_class]).dropna()
recur_df['negative'].loc[:99, '실제'] = 'negative'
recur_labeled_df = pd.concat([recur_df[c] for c in target_class]).dropna()

In [ ]:
recur_merged_df = recur_labeled_df.merge(merged_all[['index','성별','나이구간','암종','병합암종']], on = 'index', how = 'left')
metas_merged_df = metas_labeled_df.merge(merged_all[['index','성별','나이구간','암종','병합암종']], on = 'index', how = 'left')
recur_merged_df['나이구간'] = recur_merged_df['나이구간'].apply(lambda x: age_dict[x])
# recur_merged_df['암종'] = recur_merged_df['암번호'].apply(lambda x: cancer_en_dict[x])
# recur_merged_df['병합암종'] = recur_merged_df['암번호'].apply(lambda x: cnacer_merged_en_dict[x])
metas_merged_df['나이구간'] = metas_merged_df['나이구간'].apply(lambda x: age_dict[x])
# metas_merged_df['암종'] = metas_merged_df['암번호'].apply(lambda x: cancer_en_dict[x])
# metas_merged_df['병합암종'] = metas_merged_df['암번호'].apply(lambda x: cnacer_merged_en_dict[x])

In [ ]:
def welch_t2_test(X1: np.ndarray, X2: np.ndarray, alpha: float = 0.05):
    """
    두 독립 표본에 대한 Welch's T^2 검정을 수행합니다 (등분산 가정 불필요).

    Args:
        X1 (np.ndarray): 첫 번째 그룹의 데이터 행렬 (n1 x p).
        X2 (np.ndarray): 두 번째 그룹의 데이터 행렬 (n2 x p).
        alpha (float): 유의 수준 (기본값 0.05).

    Returns:
        dict: T2 통계량, F 통계량, p-value, 유의성 여부 등을 포함하는 딕셔너리.
    """
    
    # 1. 기본 통계량 계산 및 유효성 검사
    n1, p1 = X1.shape
    n2, p2 = X2.shape
    
    if p1 != p2:
        raise ValueError("두 데이터셋의 차원(p)이 일치해야 합니다.")
    p = p1  # 차원 (임베딩 벡터의 크기, 768)
    
    if n1 <= p or n2 <= p:
         # 표본 크기 < 차원인 경우, 공분산 행렬 계산이 불가능하거나 불안정해집니다.
         print("경고: 최소한 하나의 그룹에서 샘플 크기(n)가 차원(p)보다 충분히 크지 않아 검정 결과의 신뢰도가 낮거나 오류가 발생할 수 있습니다.")

    mean1 = np.mean(X1, axis=0)
    mean2 = np.mean(X2, axis=0)
    mean_diff = mean1 - mean2
    
    # 2. 개별 샘플 공분산 행렬 (S1, S2) 계산
    # rowvar=False: 각 열(column)이 변수(p)를 나타냄
    S1 = np.cov(X1, rowvar=False, ddof=1)
    S2 = np.cov(X2, rowvar=False, ddof=1)
    
    # 3. W 행렬 (가중 공분산) 및 역행렬 계산
    W1 = S1 / n1
    W2 = S2 / n2
    W = W1 + W2
    
    try:
        W_inv = np.linalg.inv(W)
    except np.linalg.LinAlgError:
        return {"error": "W 행렬의 역행렬을 계산할 수 없습니다 (행렬이 특이 행렬). 데이터에 선형 종속성이 있을 수 있습니다."}

    # 4. Welch's T^2 통계량 계산
    # T^2_W = (mean1-mean2).T * W_inv * (mean1-mean2)
    T2_statistic = mean_diff.T @ W_inv @ mean_diff
    
    # 5. 근사 자유도(df_approx, v) 계산 (Satterthwaite 근사 공식 사용)
    
    # W_inv * W_i 행렬 곱
    W_inv_W1 = W_inv @ W1
    W_inv_W2 = W_inv @ W2

    # Tr(W_inv * Wi)
    tr_W_inv_W1 = np.trace(W_inv_W1)
    tr_W_inv_W2 = np.trace(W_inv_W2)
    
    # Tr((W_inv * Wi)^2)
    tr_sq_W_inv_W1 = np.trace(W_inv_W1 @ W_inv_W1)
    tr_sq_W_inv_W2 = np.trace(W_inv_W2 @ W_inv_W2)
    
    # 근사 자유도 분모 항 계산
    denominator_term_1 = (1 / (n1 - 1)) * (tr_sq_W_inv_W1 + tr_W_inv_W1**2)
    denominator_term_2 = (1 / (n2 - 1)) * (tr_sq_W_inv_W2 + tr_W_inv_W2**2)
    
    # 근사 자유도 (v)
    df_approx = (p + p**2) / (denominator_term_1 + denominator_term_2)
    
    # 6. F 통계량으로 변환 및 p-value 계산
    
    # F-근사: F ~ T2_W * [ (v - p + 1) / (p * v) ]
    df1 = p                       # 분자 자유도
    df2 = df_approx - p + 1       # 분모 자유도 (근사)

    if df2 <= 0:
         return {"error": "계산된 근사 분모 자유도가 0 이하입니다. (df2 <= 0)", "df_approx": df_approx}
         
    F_statistic = T2_statistic * df2 / (p * df_approx)

    # p-value 계산
    p_value = 1 - f.cdf(F_statistic, df1, df2)
    
    # 7. 결과 정리
    is_significant = p_value < alpha
    
    return {
        "T2_statistic": T2_statistic,
        "F_statistic": F_statistic,
        "p_value": p_value,
        "df1": df1,
        "df2_approx": df2,
        "is_significant": is_significant,
        "alpha": alpha
    }




@torch.no_grad()
def embed_texts_with_custom_model(texts, tokenizer, model, batch_size=128, device='cuda' if torch.cuda.is_available() else 'cpu'):
    model = model.to(device)
    model.eval()
    embeddings = []
    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = list(texts[start:start+batch_size])
        encoded_input = tokenizer(batch_texts, padding=True, truncation=True, return_tensors='pt', max_length=128)
        encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
        # 모델의 encoder로부터 임베딩 추출
        outputs = model(
            input_ids=encoded_input["input_ids"],
            attention_mask=encoded_input.get("attention_mask", None),
        )
        # Pooling 방식 선택 (pooler_output이 있으면 사용, 없으면 first token)
        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            pooled = outputs.pooler_output
        else:
            pooled = outputs.last_hidden_state[:, 0, :]
        batch_embeddings = pooled.detach().cpu()
        embeddings.append(batch_embeddings)
    embeddings = torch.cat(embeddings, dim=0)
    return embeddings



model_dict = dict(
    # biobert = 'biobert-v1.1', # Done
    # biomednpl = 'BiomedNLP-PubMedBERT-Base-Uncased-Abstract-Fulltext-109M',      # model_type 설정 문제로 사용 불가
    # pubmedbert = 'PubMedBERT-base-uncased-sts-combined',
    Multilingual_E5_base = 'multilingual-e5-base',
    # medical_gemma = 'Medical-Diagnosis-COT-Gemma3-270M',
    # otaspire = 'ot-aspire-biomed-modern-bert', # Done
    # medical_doctor_ai = 'Medical_Doctor_AI_LoRA-Mistral-7B-Instruct_FullModel'   # 7B 깡모델임
)
model_dir_path = os.path.join('..','..','..','..','model')

for model_name, model_path in model_dict.items():
    break



tokenizer = AutoTokenizer.from_pretrained(os.path.join(model_dir_path, model_path))
language_model = AutoModel.from_pretrained(
        os.path.join(model_dir_path, model_path),
        # dtype="bfloat16",
    )
print(model_name, model_path, )
print('params : ', language_model.num_parameters())
print(language_model.config.hidden_size)


# save_dict = save_dir_setup(model_name, prefix=save_dir_prefix, version=save_dir_version, date = None)

In [ ]:

df = pd.read_excel([i for i in glob(os.path.join(agg_df_save_dir, '*')) if '.xlsx' in i][0])
df['나이구간'] = df['검사나이'].apply(age_group)

In [ ]:
target = 'metas'
target_word = 'Metastasis' if target == 'metas' else 'Recurrence'


In [ ]:

merged_df = df.loc[~df[target_word+'_class'].isna()]
unique_merged_df = merged_df.drop_duplicates([target_word+'_text'])

text_embeddings = embed_texts_with_custom_model(unique_merged_df[target_word+'_text'].astype(str).tolist(), tokenizer, language_model)

manifold = UMAP()
manifold_vectors = manifold.fit_transform(text_embeddings)

unique_merged_df[['umap_x1','umap_x2']] = manifold_vectors
unique_merged_df = unique_merged_df.rename(columns = {'나이구간' : 'Age','암종' : 'Cancer type','성별' : 'Sex'})
unique_merged_df.loc[:, 'Age'] = unique_merged_df.Age.apply(lambda x: age_dict[x])
# merged_df.loc[:, 'Cancer type'] = merged_df['Cancer type'].apply(lambda x: cancer_en_dict[x])

if target == 'metas':
    rev_df = metas_merged_df.copy()
else:
    rev_df = recur_merged_df.copy()
    
merged_rev_samples = pd.merge(
    rev_df[['index','prep_text','실제']],
    unique_merged_df.reset_index()[['index',target_word+'_text', 'umap_x1','umap_x2']].set_axis(['index','prep_text','umap_x1','umap_x2'], axis = 1),
    on = ['index', 'prep_text'],
    how = 'left'
)

In [ ]:
cmap = {'negative':'tab:blue', 'uncertain':'tab:green', 'positive':'tab:red', }
fig = sns.jointplot(
    data = unique_merged_df,
    x = 'umap_x1', 
    y = 'umap_x2', 
    hue = target_word+'_class', 
    kind = 'scatter',
    # palette = 'husl',
    s = 15,     # 원 크기를 더 작게 설정
    alpha = 0.5, # 반투명도 추가, 
    palette = cmap,
    # edgecolor = 'k'
    # edge_color = 'k'
    hue_order = list(cmap.keys())
)

for c in cmap.keys():
    temp_df = merged_rev_samples.loc[merged_rev_samples['실제'] == c]
    fig.ax_joint.plot(temp_df['umap_x1'], temp_df['umap_x2'], 'o', alpha=0.6, color=cmap[c], markeredgecolor='k')

# legend의 marker들의 alpha를 1로 변경
legend = fig.ax_joint.get_legend()
if legend is not None:
    for handle in legend.legend_handles:
        # scatter인 경우 alpha 속성 세팅 시도
        if hasattr(handle, 'set_alpha'):
            handle.set_alpha(1)
legend.set_title(target_word.capitalize())
if legend is not None:
    new_labels = [text.get_text().capitalize() for text in legend.texts]
    for text, new_label in zip(legend.texts, new_labels):
        text.set_text(new_label)


fig.ax_joint.set_xlabel('UMAP 1')
fig.ax_joint.set_ylabel('UMAP 2')
fig.figure.tight_layout()
fig.figure.savefig(os.path.join(figdir, f'Embedding_{target}_umap.png'), dpi = 300)

In [ ]:
# hue = 'Cancer type'
for hue in ['Sex', 'Age', 'Cancer type']:
    if hue != 'Sex':
        if hue == 'Age':
            order = age_order
        elif hue == 'Cancer type':
            order = cancer_order
    else:
        order = None

    fig = sns.jointplot(
        data = unique_merged_df, 
        x = 'umap_x1', 
        y = 'umap_x2', 
        hue = hue, 
        kind = 'scatter',
        palette = 'husl',
        s = 15,     # 원 크기를 더 작게 설정
        alpha = 0.6, # 반투명도 추가, 
        hue_order = order
    )

    fig.ax_joint.plot(merged_rev_samples['umap_x1'], merged_rev_samples['umap_x2'], 'k.', alpha = 0.6)

    fig.ax_joint.set_xlabel('UMAP 1')
    fig.ax_joint.set_ylabel('UMAP 2')
    if hue == 'Cancer type':
        fig.ax_joint.legend(ncol = 3)
    fig.figure.tight_layout()
    # fig.figure.savefig(os.path.join(figdir, f'Embedding_{target}_{hue}_umap.png'), dpi = 300)

In [ ]:
import numpy as np
from scipy.spatial.distance import pdist, squareform
from scipy.stats import f

# ----------------------------------------------------
# 1. 커널 함수 및 MMD 통계량 계산 함수 정의
# ----------------------------------------------------

def rbf_kernel_safe(X, Y, gamma):
    """
    RBF 커널 행렬 Kxy, Kx, Ky를 개별적으로 계산합니다.
    """
    # Kxy (n x m)
    sq_dists_xy = squareform(pdist(np.vstack([X, Y]), 'sqeuclidean'))
    Kxy = np.exp(-gamma * sq_dists_xy[:X.shape[0], X.shape[0]:])
    
    # Kx (n x n)
    sq_dists_x = squareform(pdist(X, 'sqeuclidean'))
    Kx = np.exp(-gamma * sq_dists_x)
    
    # Ky (m x m)
    sq_dists_y = squareform(pdist(Y, 'sqeuclidean'))
    Ky = np.exp(-gamma * sq_dists_y)
    
    return Kx, Ky, Kxy

def calculate_mmd_safe(Kx, Ky, Kxy, n, m):
    """
    개별 커널 행렬을 사용하여 MMD^2 통계량을 계산합니다.
    """
    term_xx = np.sum(Kx) / (n * n)
    term_yy = np.sum(Ky) / (m * m)
    term_xy = np.sum(Kxy) * 2 / (n * m)
    
    mmd_sq = term_xx + term_yy - term_xy
    return mmd_sq

# ----------------------------------------------------
# 2. Subsample 기반 MMD Test 함수 정의
# ----------------------------------------------------

def mmd_subsample_test(X1_large: np.ndarray, X2_small: np.ndarray, 
                       gamma: float = None, n_permutations: int = 500, alpha: float = 0.05):
    """
    대규모 그룹 X1에서 소규모 그룹 X2와 동일한 크기를 추출하여 MMD 검정을 수행합니다.
    (메모리 효율성을 위해 대규모 그룹의 대표성을 희생합니다.)
    """
    
    n_large, p = X1_large.shape
    n_small, _ = X2_small.shape
    
    # 1. 균형 잡힌 표본 추출 (n1 = n2 = m)
    # X1_large에서 n_small만큼 무작위 추출
    np.random.seed(42) # 재현성을 위해 시드 설정
    sample_indices = np.random.choice(n_large, size=n_small, replace=False)
    X1_subsampled = X1_large[sample_indices, :]
    
    n, m = X1_subsampled.shape[0], X2_small.shape[0] # n=m=300
    N = n + m
    
    # 2. Gamma 값 설정 (Median Heuristic)
    combined_small_data = np.vstack([X1_subsampled, X2_small])
    if gamma is None:
        all_dists_sq = pdist(combined_small_data, 'sqeuclidean')
        median_dist = np.median(all_dists_sq)
        gamma = 1.0 / median_dist if median_dist > 0 else 1.0 
    
    # 3. 실제 MMD^2 통계량 계산
    Kx_real, Ky_real, Kxy_real = rbf_kernel_safe(X1_subsampled, X2_small, gamma)
    mmd_real = calculate_mmd_safe(Kx_real, Ky_real, Kxy_real, n, m)
    
    # 4. Permutation Test 수행
    mmd_null_distribution = []
    
    # K matrix를 구성하지 않고, Kx, Ky, Kxy를 재배열하여 통계량만 계산
    # Kx, Ky, Kxy를 합쳐서 N x N K 행렬을 만들면 다시 메모리 문제가 발생하므로,
    # 순열 테스트를 위해 Kx, Ky, Kxy를 사용하여 통계량만 계산합니다.
    
    # 300x300 행렬만 메모리에 올립니다.
    Kx_real_flat = Kx_real.flatten()
    Ky_real_flat = Ky_real.flatten()
    Kxy_real_flat = Kxy_real.flatten()
    
    # 합쳐진 데이터 (Kx, Ky, Kxy)의 통계량 계산을 위한 플랫 배열
    # K_full_flat을 만드는 대신, 필요한 요소만 사용하여 계산해야 하지만,
    # 순열 검정의 표준 구현은 전체 커널 행렬 K를 섞는 방식입니다.
    # 메모리 문제 때문에 순열 검정 횟수(n_permutations)를 낮추는 것이 현실적입니다.
    
    # 임시로 전체 커널 행렬 K를 구성하여 셔플링을 수행합니다.
    # 만약 600x600 행렬에서도 문제가 발생한다면, Permutation 횟수를 더 낮춰야 합니다.
    K_full = np.block([[Kx_real, Kxy_real], 
                       [Kxy_real.T, Ky_real]])
    
    indices = np.arange(N)
    
    for _ in range(n_permutations):
        # 인덱스를 무작위로 섞음
        np.random.shuffle(indices)
        
        # 섞인 인덱스를 바탕으로 K 행렬에서 새로운 Kx, Ky, Kxy 부분을 추출
        K_perm = K_full[indices, :][:, indices] 
        
        # 순열된 데이터에 대한 MMD^2 계산
        mmd_perm = calculate_mmd_safe(K_perm[:n, :n], K_perm[n:, n:], K_perm[:n, n:], n, m)
        mmd_null_distribution.append(mmd_perm)
        
    mmd_null_distribution = np.array(mmd_null_distribution)
    p_value = np.sum(mmd_null_distribution >= mmd_real) / n_permutations
    
    is_significant = p_value < alpha
    
    return {
        "mmd_sq_statistic": mmd_real,
        "gamma_bandwidth": gamma,
        "p_value": p_value,
        "n_permutations": n_permutations,
        "subsample_size": n_small,
        "is_significant": is_significant,
        "alpha": alpha
    }

In [ ]:
rev_idx = unique_merged_df.index.isin(rev_df['index'])

# Get indices for rev and no-rev
rev_indices = unique_merged_df.index[rev_idx]
no_rev_indices = unique_merged_df.index[~rev_idx]

rev_embeddings = text_embeddings[rev_idx].numpy()
# rev_embeddings = manifold_vectors[rev_idx]

rev_df_sorted = rev_df.set_index('index').loc[rev_indices]

no_rev_embeddings = text_embeddings[~rev_idx].numpy()
# no_rev_embeddings = manifold_vectors[~rev_idx]
# Ensure that the order of no_rev_df_sorted matches the order of no_rev_embeddings (i.e. no_rev_indices)
no_rev_df_sorted = unique_merged_df.loc[no_rev_indices]

# rev_label = 
# no_rev_embeddings = text_embeddings[~merged_df.index.isin(rev_df['index'])].numpy()
# load_all['Metastasis']

In [ ]:
c = 'positive'

In [ ]:

for c in ['negative','uncertain','positive']:
    # print(c)
    # print(welch_t2_test(
    #     no_rev_embeddings[no_rev_df_sorted[target_word+'_class'].apply(lambda x: x == c)], 
    #     rev_embeddings[rev_df_sorted['실제'] == c], 
    # ))
    print(mmd_subsample_test(
        no_rev_embeddings[no_rev_df_sorted[target_word+'_class'].apply(lambda x: x == c)], 
        rev_embeddings[rev_df_sorted['실제'] == c], 
    ))

In [ ]:
import numpy as np
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.stats import norm

# ----------------------------------------------------
# 1. 커널 함수 (RBF Kernel)
# ----------------------------------------------------

def rbf_kernel_matrix(X, Y, gamma):
    """
    RBF 커널 행렬 K(X, Y)를 계산합니다.
    """
    # X와 Y 간의 유클리디안 거리 제곱
    sq_dists = cdist(X, Y, 'sqeuclidean')
    # K(x, y) = exp(-gamma * ||x - y||^2)
    return np.exp(-gamma * sq_dists)

# ----------------------------------------------------
# 2. 선형 MMD (MMD_L) 및 분산 (Variance) 계산 함수
# ----------------------------------------------------

def calculate_mmd_linear(K_XX, K_YY, K_XY, n_blocks):
    """
    K_XX, K_YY, K_XY 커널 행렬을 사용하여 선형 MMD^2 통계량을 계산합니다.
    K_XX, K_YY는 대각선 요소가 0인 형태여야 U-통계량을 정확히 추정합니다.
    """
    
    # 1. MMD^2 선형 통계량 (V-statistic 추정량)
    # K_XX (n x n), K_YY (m x m), K_XY (n x m)
    
    # 2. MMD^2 계산
    # MMD_L = mean(K_XX) + mean(K_YY) - 2 * mean(K_XY)
    
    # K_XX, K_YY에서 대각선 요소를 제외 (U-statistic의 불편 추정량을 따름)
    # n*(n-1)로 나누는 것이 엄밀한 U-통계량이나, 여기서는 V-통계량 추정으로 단순화
    
    term_xx = np.sum(K_XX - np.diag(np.diag(K_XX))) / (n_blocks * (n_blocks - 1))
    term_yy = np.sum(K_YY - np.diag(np.diag(K_YY))) / (n_blocks * (n_blocks - 1))
    term_xy = np.sum(K_XY) / (n_blocks * n_blocks)
    
    mmd_sq_linear = term_xx + term_yy - 2 * term_xy
    return mmd_sq_linear

# ----------------------------------------------------
# 3. MMD Linear Test (메인 함수)
# ----------------------------------------------------

def mmd_linear_test(X1: np.ndarray, X2: np.ndarray, 
                    n_blocks: int = 300, n_permutations: int = 500, alpha: float = 0.05):
    """
    대규모 데이터셋에 대한 선형 시간 MMD 검정을 수행합니다.
    X1과 X2에서 각각 n_blocks 크기의 데이터를 무작위 추출하여 MMD를 근사합니다.
    """
    n1, p = X1.shape
    n2, _ = X2.shape
    
    if n_blocks > min(n1, n2):
        # raise ValueError(f"n_blocks ({n_blocks})는 min(n1, n2)={min(n1, n2)}보다 작거나 같아야 합니다.")
        n_blocks = min(n1, n2)

    # 1. 균형 잡힌 표본 추출 (n = m = n_blocks)
    np.random.seed(42)
    
    # 2. Gamma 값 설정 (Median Heuristic for Subsample)
    # 전체 데이터에서 Gamma를 계산할 수 없으므로, 초기 샘플을 추출하여 설정
    X1_sample_init = X1[np.random.choice(n1, n_blocks, replace=False), :]
    X2_sample_init = X2[np.random.choice(n2, n_blocks, replace=False), :]
    combined_sample = np.vstack([X1_sample_init, X2_sample_init])
    
    all_dists_sq = pdist(combined_sample, 'sqeuclidean')
    median_dist = np.median(all_dists_sq)
    gamma = 1.0 / median_dist if median_dist > 0 else 1.0 
    
    # 3. 실제 MMD^2 통계량 계산 (Subsample)
    K_XX_real = rbf_kernel_matrix(X1_sample_init, X1_sample_init, gamma)
    K_YY_real = rbf_kernel_matrix(X2_sample_init, X2_sample_init, gamma)
    K_XY_real = rbf_kernel_matrix(X1_sample_init, X2_sample_init, gamma)
    
    mmd_real = calculate_mmd_linear(K_XX_real, K_YY_real, K_XY_real, n_blocks)
    
    # 4. Permutation Test 수행 (Null Distribution 구축)
    mmd_null_distribution = []
    
    # 합쳐진 샘플 데이터 (2 * n_blocks)
    X_combined_sample = combined_sample
    
    for _ in range(n_permutations):
        # 인덱스 무작위 셔플링
        indices = np.random.permutation(X_combined_sample.shape[0])
        X_perm = X_combined_sample[indices, :]
        
        # X1_perm과 X2_perm으로 나누기
        X1_perm = X_perm[:n_blocks, :]
        X2_perm = X_perm[n_blocks:, :]
        
        # 섞인 그룹에 대한 MMD 통계량 계산
        K_XX_perm = rbf_kernel_matrix(X1_perm, X1_perm, gamma)
        K_YY_perm = rbf_kernel_matrix(X2_perm, X2_perm, gamma)
        K_XY_perm = rbf_kernel_matrix(X1_perm, X2_perm, gamma)

        mmd_perm = calculate_mmd_linear(K_XX_perm, K_YY_perm, K_XY_perm, n_blocks)
        mmd_null_distribution.append(mmd_perm)
        
    mmd_null_distribution = np.array(mmd_null_distribution)
    p_value = np.sum(mmd_null_distribution >= mmd_real) / n_permutations
    
    is_significant = p_value < alpha
    
    return {
        "mmd_sq_statistic": mmd_real,
        "gamma_bandwidth": gamma,
        "p_value": p_value,
        "n_permutations": n_permutations,
        "n_blocks": n_blocks,
        "is_significant": is_significant,
        "alpha": alpha
    }

In [ ]:

for c in ['negative','uncertain','positive']:
    # print(c)
    # print(welch_t2_test(
    #     no_rev_embeddings[no_rev_df_sorted[target_word+'_class'].apply(lambda x: x == c)], 
    #     rev_embeddings[rev_df_sorted['실제'] == c], 
    # ))
    print(mmd_linear_test(
        no_rev_embeddings[no_rev_df_sorted[target_word+'_class'].apply(lambda x: x == c)], 
        rev_embeddings[rev_df_sorted['실제'] == c], 
    ))

In [ ]:
import numpy as np
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.stats import norm

# ----------------------------------------------------
# 1. 커널 함수 (RBF Kernel)
# ----------------------------------------------------

def rbf_kernel_matrix(X, Y, gamma):
    """
    RBF 커널 행렬 K(X, Y)를 계산합니다.
    """
    # X와 Y 간의 유클리디안 거리 제곱
    sq_dists = cdist(X, Y, 'sqeuclidean')
    # K(x, y) = exp(-gamma * ||x - y||^2)
    return np.exp(-gamma * sq_dists)

# ----------------------------------------------------
# 2. 선형 MMD (MMD_L) 및 분산 (Variance) 계산 함수
# ----------------------------------------------------

def calculate_mmd_linear(K_XX, K_YY, K_XY, n_blocks):
    """
    K_XX, K_YY, K_XY 커널 행렬을 사용하여 선형 MMD^2 통계량을 계산합니다.
    K_XX, K_YY는 대각선 요소가 0인 형태여야 U-통계량을 정확히 추정합니다.
    """
    
    # 1. MMD^2 선형 통계량 (V-statistic 추정량)
    # K_XX (n x n), K_YY (m x m), K_XY (n x m)
    
    # 2. MMD^2 계산
    # MMD_L = mean(K_XX) + mean(K_YY) - 2 * mean(K_XY)
    
    # K_XX, K_YY에서 대각선 요소를 제외 (U-statistic의 불편 추정량을 따름)
    # n*(n-1)로 나누는 것이 엄밀한 U-통계량이나, 여기서는 V-통계량 추정으로 단순화
    
    term_xx = np.sum(K_XX - np.diag(np.diag(K_XX))) / (n_blocks * (n_blocks - 1))
    term_yy = np.sum(K_YY - np.diag(np.diag(K_YY))) / (n_blocks * (n_blocks - 1))
    term_xy = np.sum(K_XY) / (n_blocks * n_blocks)
    
    mmd_sq_linear = term_xx + term_yy - 2 * term_xy
    return mmd_sq_linear

# ----------------------------------------------------
# 3. MMD Linear Test (메인 함수)
# ----------------------------------------------------

def modified_mmd_linear_test(X1: np.ndarray, X2: np.ndarray, 
                    n_blocks: int = 300, n_permutations: int = 1000, alpha: float = 0.05):
    """
    대규모 데이터셋에 대한 선형 시간 MMD 검정을 수행합니다.
    X1과 X2에서 각각 n_blocks 크기의 데이터를 무작위 추출하여 MMD를 근사합니다.
    """
    n1, p = X1.shape
    n2, _ = X2.shape
    
    if n_blocks > min(n1, n2):
        # raise ValueError(f"n_blocks ({n_blocks})는 min(n1, n2)={min(n1, n2)}보다 작거나 같아야 합니다.")
        n_blocks = min(n1, n2)

    # 1. 균형 잡힌 표본 추출 (n = m = n_blocks)
    # np.random.seed(42)
    
    # 4. Permutation Test 수행 (Null Distribution 구축)
    mmd_null_distribution = []
    
    
    for _ in tqdm(range(n_permutations)):
            
        # 2. Gamma 값 설정 (Median Heuristic for Subsample)
        # 전체 데이터에서 Gamma를 계산할 수 없으므로, 초기 샘플을 추출하여 설정
        X1_sample_init = X1[np.random.choice(n1, n_blocks, replace=False), :]
        X2_sample_init = X2[np.random.choice(n2, n_blocks, replace=False), :]
        combined_sample = np.vstack([X1_sample_init, X2_sample_init])
        
        all_dists_sq = pdist(combined_sample, 'sqeuclidean')
        median_dist = np.median(all_dists_sq)
        gamma = 1.0 / median_dist if median_dist > 0 else 1.0 
        
        # 3. 실제 MMD^2 통계량 계산 (Subsample)
        K_XX_real = rbf_kernel_matrix(X1_sample_init, X1_sample_init, gamma)
        K_YY_real = rbf_kernel_matrix(X2_sample_init, X2_sample_init, gamma)
        K_XY_real = rbf_kernel_matrix(X1_sample_init, X2_sample_init, gamma)
        
        mmd_real = calculate_mmd_linear(K_XX_real, K_YY_real, K_XY_real, n_blocks)

        # 합쳐진 샘플 데이터 (2 * n_blocks)
        X_combined_sample = combined_sample


        # 인덱스 무작위 셔플링
        indices = np.random.permutation(X_combined_sample.shape[0])
        X_perm = X_combined_sample[indices, :]
        
        # X1_perm과 X2_perm으로 나누기
        X1_perm = X_perm[:n_blocks, :]
        X2_perm = X_perm[n_blocks:, :]
        
        # 섞인 그룹에 대한 MMD 통계량 계산
        K_XX_perm = rbf_kernel_matrix(X1_perm, X1_perm, gamma)
        K_YY_perm = rbf_kernel_matrix(X2_perm, X2_perm, gamma)
        K_XY_perm = rbf_kernel_matrix(X1_perm, X2_perm, gamma)

        mmd_perm = calculate_mmd_linear(K_XX_perm, K_YY_perm, K_XY_perm, n_blocks)
        mmd_null_distribution.append(mmd_perm >= mmd_real)

    mmd_null_distribution = np.array(mmd_null_distribution)
    p_value = np.sum(mmd_null_distribution) / n_permutations
    


    #     mmd_null_distribution.append(mmd_perm)
        
    # mmd_null_distribution = np.array(mmd_null_distribution)
    # p_value = np.sum(mmd_null_distribution >= mmd_real) / n_permutations
    
    is_significant = p_value < alpha
    
    return {
        "mmd_sq_statistic": mmd_real,
        "gamma_bandwidth": gamma,
        "p_value": p_value,
        "n_permutations": n_permutations,
        "n_blocks": n_blocks,
        "is_significant": is_significant,
        "alpha": alpha
    }

In [ ]:

for c in ['negative','uncertain','positive']:
    # print(c)
    # print(welch_t2_test(
    #     no_rev_embeddings[no_rev_df_sorted[target_word+'_class'].apply(lambda x: x == c)], 
    #     rev_embeddings[rev_df_sorted['실제'] == c], 
    # ))
    print(modified_mmd_linear_test(
        no_rev_embeddings[no_rev_df_sorted[target_word+'_class'].apply(lambda x: x == c)], 
        rev_embeddings[rev_df_sorted['실제'] == c], 
    ))